# 实验清单

## 水印嵌入

实验环境与水印导入

In [ ]:
# ============== 导入核心库 ================
from watermarks.core import WatermarkerFactory
import watermarks.implementations
import importlib
import pkgutil
from utils.monitor import WatermarkProfiler  # type: ignore

# ============== 动态加载所有水印实现模块 ================
imported_modules = {}
package = watermarks.implementations
prefix = package.__name__ + "."

for _, name, is_pkg in pkgutil.iter_modules(package.__path__, prefix):
    module = importlib.import_module(name)
    short_name = name.split('.')[-1]
    imported_modules[short_name] = module
    print(f"已导入模块: {short_name}")

print("所有水印实现模块加载完毕。")
print(f"可用模块列表: {list(imported_modules.keys())}")

# ============== 水印实验执行函数 ================
# 进行硬编码：将seed、sampling_retio、prompt_folder等参数直接写入函数中，确保每次实验的一致性
def run_watermark_experiment(method_name, method_params, input_folder, output_folder, threshold=0.05, mix_ratio=0.5, prompt_folder=None):
    """
    运行指定水印方法的嵌入和提取实验
    
    参数:
        method_name: 水印方法名称 (如 'lsb', 'tree_ring' 等)
        method_params: 该水印方法所需的参数字典
    """
    print(f"\n{'='*50}")
    print(f"开始测试水印方法: {method_name}")
    print(f"参数配置: {method_params}")
    print(f"{'='*50}")
    
    method_params['seed'] = 42
    method_params['sampling_ratio'] = 0.05
    method_params['prompt_folder'] = prompt_folder
    # 创建水印器实例
    attacker = WatermarkerFactory.create(
        name=method_name,
        params=method_params
    )
    
    # 执行水印嵌入
    print("正在执行水印嵌入...")
    with WatermarkProfiler(f"{method_name}_embedding"):
        attacker.embed_batch(
            input_dir=input_folder,
            output_dir=output_folder
        )
    print("水印嵌入完成。")
    
    # 执行水印提取与检测
    print("正在执行水印提取与检测...")
    with WatermarkProfiler(f"{method_name}_extraction"):
        result = attacker.extract_batch(
            image_dir_to_check=output_folder,
            clean_dir= input_folder,
            threshold=  threshold,
            mix_ratio = mix_ratio,
        )
    
    print(f"水印检测结果: {result}")
    print(f"{'='*50}")
    print(f"方法 {method_name} 测试完成。")
    print(f"{'='*50}\n")
    
    return result



已导入模块: cin
已导入模块: dwt
已导入模块: dwt_svd
已导入模块: lsb
已导入模块: mbrs
已导入模块: tree_ring
所有水印实现模块加载完毕。
可用模块列表: ['cin', 'dwt', 'dwt_svd', 'lsb', 'mbrs', 'tree_ring']


### LSB嵌入

实验配置：

In [2]:
# 路径参数
input_folder = "D:/graduation/computer/Watermark/dataset/origin"
output_folder = "D:/graduation/computer/Watermark/results/lsb_watermarked"

SECRET_32 = [1,0,1,0, 0,1,0,1, 1,1,0,0, 0,0,1,1,
             1,0,0,1, 0,1,1,0, 1,1,1,0, 0,0,0,1]

In [3]:
print("执行LSB水印实验...")
lsb_params = {
    "secret": SECRET_32
}
lsb_result = run_watermark_experiment(
    method_name="lsb",
    method_params=lsb_params,
    input_folder=input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5
)
print("LSB水印实验完成。")

执行LSB水印实验...

开始测试水印方法: lsb
参数配置: {'secret': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1]}
正在执行水印嵌入...
--- [Profiler] Start monitoring: lsb_embedding ---
[LSB] Embedding start. Processing 20000 files...
--- [Profiler] Result for lsb_embedding ---
    Max RAM Usage : 664.20 MB
    Max VRAM Usage: 51.94 MB
-----------------------------------------


KeyboardInterrupt: 